# 10주차 ③ 어텐션의 도입과 시각화 — 실습 5~7  〔빈칸본〕

> **빈칸이 3곳입니다.** 전부 셀 1 의 `Attention.forward` 안에 있고,
> **점수 → softmax → 가중합** 세 단계가 그대로 빈칸 ①②③ 입니다.
> 이 세 줄이 **다음 주 트랜스포머에서도 그대로** 나옵니다.
> 다 채운 노트북은 `19_attention.ipynb` 로 저장해 제출합니다.

**목표**: 마지막 은닉 상태 하나로 부족한 이유를 이해하고,
**점수 → softmax → 가중합** 세 단계로 어텐션을 구현해
**모델이 어느 단어를 봤는지** 눈으로 확인한다.

> **과제 제출 대상 노트북입니다.**

```python
out, (h, c) = self.lstm(e)      # out : (B, 40, 128)   ← 40개를 다 계산해 놓고
last = h[-1]                    #                        마지막 1개만 썼다  ★
```

```
   "이 영화는 배우도 좋고 영상미도 훌륭하고 음악도 인상적인데 결말이 최악이다"

     h₁    h₂    h₃   ...   h₃₈   h₃₉   h₄₀
      ↑                             ↑     ↑
    "이"                        "최악"  "이다"
                                        └─ 우리는 이것만 썼다

   판정에 결정적인 건 h₃₉ 근처인데, h₄₀ 안에서 앞의 칭찬들과 뭉개져 있다
```

> **핵심 질문 ★★**: **문장 전체를 벡터 하나에 욱여넣는 게 맞을까요?**
> 40단어를 128차원 벡터 **하나**로 요약해야 합니다. 정보가 손실될 수밖에 없습니다.
> 게다가 1교시에서 본 대로, 문장이 짧으면 **뒤쪽은 대부분 패딩**입니다.

```
   [기존]  40개를 계산했지만 마지막 1개만 쓴다

   [어텐션]  40개를 다 남겨 두고,  어느 것을 얼마나 볼지 "가중치"를 학습한다  ★

        h₁   h₂   h₃  ...  h₃₈  h₃₉  h₄₀
        0.01 0.02 0.01     0.05 0.62 0.03      ← 가중치 (합 = 1)
          ↓    ↓    ↓        ↓    ↓    ↓
        ────────── 가중합 ──────────→  문맥 벡터 → 분류
                                          ↑
                              "최악"에 0.62 를 준다면 잘 배운 것
```

### 어텐션의 계산 — 세 단계뿐이다

```
   ① 점수      score = v · tanh(W·hᵢ)              → (B, 40)
   ② softmax   α = softmax(score)                   ★ 합 = 1  → (B, 40)
   ③ 가중합    context = Σ αᵢ · hᵢ                  → (B, 128)
```

> **핵심 메시지 ★ (출제 지점)**: **softmax 를 쓰는 이유는 합을 1로 만들기 위해서**입니다.
> 그래야 *"이 단어에 62%를 줬다"* 처럼 읽을 수 있습니다.

```
   "재미없다" 뒤의 <pad> 36개에도 점수가 매겨진다  →  그대로 두면 패딩에 주목할 수 있다
        해법 : 패딩 위치의 점수를 -inf 로 만든 뒤 softmax   →  softmax(-inf) = 0
```

> **마스킹(masking)** 이라고 합니다. **11주차 트랜스포머에서 다시, 더 중요하게** 나옵니다.

## 실습 5 — 어텐션 층 추가 ★

In [ ]:
# 셀 0 — 앞 교시에서 이어서 (커널을 재시작했다면 이 셀부터)
import torch, torch.nn as nn, time, os, json
from torch.utils.data import TensorDataset, DataLoader
from preprocess_text import load_nsmc, load_vocab, encode, invert_vocab

device = "cuda" if torch.cuda.is_available() else "cpu"
train_texts, train_labels, val_texts, val_labels = load_nsmc("data/nsmc_subset")
vocab = load_vocab("models/vocab.json")
VOCAB_SIZE, EMBED, HIDDEN, MAX_LEN = len(vocab), 128, 128, 40

X  = torch.tensor([encode(t, vocab, MAX_LEN) for t in train_texts])
y  = torch.tensor(train_labels)
Xv = torch.tensor([encode(t, vocab, MAX_LEN) for t in val_texts])
yv = torch.tensor(val_labels)
train_loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(Xv, yv), batch_size=128)

lstm_acc = json.load(open("results/lstm.json"))["lstm_acc"]     # 2교시 결과
print(f"장치 {device} | 2교시 LSTM 정확도 {lstm_acc*100:.2f}%")

In [ ]:
# 셀 1 — 어텐션이 붙은 모델
class Attention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.W = nn.Linear(hidden, hidden)
        self.v = nn.Linear(hidden, 1, bias=False)

    def forward(self, out, mask):
        # out  : (B, L, H)    모든 시점의 은닉 상태
        # mask : (B, L)       True = 실제 토큰, False = 패딩

        # ───── 빈칸 ① : 점수 계산 (한 줄) ─────
        # 힌트:  score = self.v(torch.tanh(self.W(out))).squeeze(-1)   → (B, L)


        # ───── 빈칸 ② : 패딩 위치를 -inf 로 (한 줄) ─────
        # 힌트:  score = score.masked_fill(~mask, float("-inf"))


        # ───── 빈칸 ③ : softmax 로 가중치를 만들고 가중합 (두 줄) ─────
        # 힌트:  alpha = torch.softmax(score, dim=__)                  → (B, L)
        #        context = (out * alpha.unsqueeze(-1)).sum(dim=__)     → (B, H)


        return context, alpha


class LSTMAttnClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb  = nn.Embedding(VOCAB_SIZE, EMBED, padding_idx=0)
        self.lstm = nn.LSTM(EMBED, HIDDEN, batch_first=True)
        self.attn = Attention(HIDDEN)
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(HIDDEN, 2)

    def forward(self, x, return_attn=False):
        mask = (x != 0)                            # ★ 패딩이 아닌 곳
        e = self.emb(x)
        out, _ = self.lstm(e)                      # ★ 이번엔 out 을 쓴다 (h 가 아니라)
        context, alpha = self.attn(out, mask)
        logits = self.fc(self.drop(context))
        return (logits, alpha) if return_attn else logits

torch.manual_seed(0)
model_attn = LSTMAttnClassifier().to(device)
dummy = torch.randint(1, 100, (4, MAX_LEN)).to(device)
lo, al = model_attn(dummy, return_attn=True)
print("logits :", lo.shape, "| alpha :", al.shape)      # (4,2) / (4,40)
print("alpha 합 (1이어야 함) :", al.sum(dim=1))

> **핵심 메시지 ★★**: 2교시 모델과 비교하면 **바뀐 곳이 두 군데**입니다.
> ```
>   out, (h, c) = self.lstm(e);  last = h[-1]                   ← 2교시: 마지막 하나
>   out, _      = self.lstm(e);  context, α = attn(out, mask)   ← 오늘: 전부 쓰고 가중합
> ```
> **버렸던 `out` 을 살린 것**이 어텐션의 전부입니다.

> **관찰 포인트 ★**: `alpha` 의 합이 **정확히 1** 입니다. softmax 를 썼기 때문입니다.
> 이 값이 실습 7에서 **막대그래프**가 됩니다.

In [ ]:
# 셀 2 — 마스킹이 실제로 동작하는지 확인
x_short = torch.zeros(1, MAX_LEN, dtype=torch.long)
x_short[0, :5] = torch.tensor([3, 7, 11, 4, 9])       # 5단어 문장 + 패딩 35개
with torch.no_grad():
    _, a = model_attn(x_short.to(device), return_attn=True)

print("실제 토큰 5개의 가중치 :", a[0, :5].cpu().numpy().round(4))
print("패딩 35개의 가중치 합  :", a[0, 5:].sum().item())
print("→ 패딩 쪽이 0 이면 마스킹이 제대로 동작한 것입니다")

> **막히면**:
> | 증상 | 원인 |
> |---|---|
> | `alpha` 합이 1이 아니다 | `dim` 을 잘못 줬다. **`dim=1`**(시퀀스 축) |
> | shape 오류 | `alpha.unsqueeze(-1)` 을 빠뜨렸다. `(B,L)` → `(B,L,1)` |
> | 손실이 `nan` | 문장 전체가 패딩이면 전부 `-inf` → softmax 가 `nan`. 빈 문장을 걸러낸다 |
> | 어텐션이 패딩을 본다 | 마스킹을 안 넣었다 ★ |

## 실습 6 — 성능 비교

In [ ]:
# 셀 3 — 같은 조건으로 학습해 비교
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_attn.parameters(), lr=1e-3)
EPOCHS = 5

def evaluate(m):
    m.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (m(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t

t0 = time.time()
for epoch in range(EPOCHS):
    model_attn.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model_attn(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f"epoch {epoch+1}/{EPOCHS} | 검증 정확도 {evaluate(model_attn)*100:5.2f}%")

attn_acc = evaluate(model_attn)
print(f"\n{'':22s}{'정확도':>10s}")
print(f"{'LSTM (2교시)':22s}{lstm_acc*100:>9.2f}%")
print(f"{'LSTM + 어텐션':22s}{attn_acc*100:>9.2f}%")
print(f"{'차이':22s}{(attn_acc-lstm_acc)*100:>+9.2f}%p")
print(f"\n어텐션 층 파라미터 : {sum(p.numel() for p in model_attn.attn.parameters()):,} 개뿐")

> **관찰 포인트**: 차이가 **크지 않을 수도** 있습니다. 그래도 괜찮습니다.
> **오늘의 진짜 이득은 정확도가 아니라 다음 실습의 해석 가능성**입니다.
> 그리고 **11주차의 출발점**이라는 것이 더 중요합니다.

> **핵심 메시지 ★**: 어텐션 층은 파라미터가 **아주 적습니다**(`W`, `v` 두 개).
> 파라미터를 거의 안 늘리고 성능과 해석력을 얻습니다.

> ⚠️ 문장이 짧으면 차이가 안 납니다 — 마지막 상태로도 충분하기 때문입니다.
> **긴 리뷰가 포함된 데이터**에서 차이가 잘 보입니다.

In [ ]:
# 셀 4 — 저장
os.makedirs("models", exist_ok=True)
torch.save(model_attn.state_dict(), "models/lstm_attn.pt")
json.dump({"lstm_acc": lstm_acc, "attn_acc": attn_acc},
          open("results/attention.json", "w"), ensure_ascii=False, indent=2)
print("저장 완료 : models/lstm_attn.pt")

## 실습 7 — 어텐션 가중치 시각화 ★★

In [ ]:
# 셀 5 — 모델이 어느 단어를 봤나
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

itos = invert_vocab(vocab)                       # 인덱스 → 단어

def show_attention(text):
    idx = encode(text, vocab, MAX_LEN)
    x = torch.tensor([idx]).to(device)
    model_attn.eval()
    with torch.no_grad():
        logits, alpha = model_attn(x, return_attn=True)
    p = torch.softmax(logits, dim=1)[0]
    n = sum(1 for i in idx if i != 0)             # 실제 토큰 수
    words = [itos.get(i, "?") for i in idx[:n]]
    a = alpha[0, :n].cpu()

    plt.figure(figsize=(min(12, 0.9*n+2), 2.6))
    plt.bar(range(n), a, color=["crimson" if v == a.max() else "steelblue" for v in a])
    plt.xticks(range(n), words, rotation=45, ha="right")
    plt.ylabel("어텐션 가중치")
    plt.title(f"{text}\n→ 부정 {p[0]*100:.1f}% / 긍정 {p[1]*100:.1f}%")
    plt.tight_layout(); plt.show()

    print(f"  가장 주목한 단어 : {words[int(a.argmax())]}  ({a.max()*100:.1f}%)")

for s in ["정말 재미있었다",
          "시간이 아까웠다",
          "배우도 좋고 영상도 훌륭한데 결말이 최악이다"]:      # ★ 세 번째
    show_attention(s)

> **관찰 포인트 ★★**: 세 번째 문장에서 **빨간 막대(최댓값)가 어디에 있나요?**
> - **"최악" 위에 있다** → 모델이 결정적인 단어를 제대로 골랐습니다
> - **엉뚱한 데 있다** → 그것도 **좋은 관찰**입니다. *"왜 그랬을까"* 를 생각해 보세요

> **핵심 메시지 ★★**: 9주차 Grad-CAM 과 나란히 놓아 보세요.
> ```
>   Grad-CAM      : 이미지의 어느 "영역"을 봤나
>   어텐션 가중치 : 문장의 어느 "단어"를 봤나
> ```
> 둘 다 **블랙박스를 여는 창**이고, 둘 다 **한계가 같습니다** —
> *"어디를 봤다"* 는 알려 주지만 ***"왜 그렇게 판단했는지"*** 는 알려 주지 않습니다.

In [ ]:
# 셀 6 (여유가 있으면) — 검증셋에서 어텐션이 가장 강하게 주목한 단어들
from collections import Counter
model_attn.eval()
top_words = Counter()
with torch.no_grad():
    for xb, _ in val_loader:
        xb = xb.to(device)
        _, a = model_attn(xb, return_attn=True)
        for row_x, row_a in zip(xb.cpu(), a.cpu()):
            j = int(row_a.argmax())
            w = itos.get(int(row_x[j]), "?")
            if w not in ("<pad>", "<unk>"):
                top_words[w] += 1

print("검증셋에서 가장 자주 1등으로 주목받은 단어 20개")
for w, n in top_words.most_common(20):
    print(f"  {w:12s} {n:4d}회")

> **관찰 포인트**: 감성이 뚜렷한 단어(재밌다, 최악, 아깝다 …)가 위에 오면
> 모델이 **말이 되는 것을 배웠다**는 신호입니다.
> 엉뚱한 조사·기호가 위에 온다면 — 그것도 **과제에 쓸 좋은 관찰**입니다.

---

### 과제 (마감 11/12 목 23:59)

```
  ① 19_attention.ipynb (출력 저장)
  ② LSTM vs LSTM+어텐션 정확도 비교
  ③ 어텐션 가중치 시각화 3문장
  ④ "모델이 주목한 단어가 납득되는가" 해석 3줄     ★ 핵심
  ⑤ 회고 / 커밋 · push · LMS 제출
```

> **과제의 초점은 정확도가 아니라 ④의 해석입니다.**
> 어텐션이 **엉뚱한 단어를 봤다면 그것도 좋은 관찰**입니다.
> *"왜 그렇게 됐을까"* 를 쓰면 만점입니다 — 예를 들면
> *"학습 데이터에 '결말'이라는 단어가 부정 리뷰에 자주 나와서일 수 있다"* 같은 추론입니다.

### 11주차 예고 ★

> 오늘 어텐션을 LSTM 에 **붙였습니다**. 다음 주의 질문은 이겁니다 —
> **"그럼 LSTM 을 빼고 어텐션만 남기면 어떨까?"**
>
> ```
>   LSTM 의 문제 : 순차적으로 읽어야 한다 → 느리다. 그리고 여전히 멀면 흐려진다
>   어텐션       : 모든 시점을 한 번에 본다 → 거리와 무관. 병렬 처리도 가능
>
>        → 순환을 아예 없애고 어텐션만 쌓는다  =  트랜스포머  ★
> ```
>
> 오늘 `alpha` 를 계산한 그 세 줄이 **거기서도 그대로** 나옵니다.

### 이 노트북 체크리스트

- [ ] 마지막 은닉 상태 하나로 부족한 이유를 예문으로 설명할 수 있다 ★★
- [ ] 어텐션 계산 세 단계를 말할 수 있다 ★
- [ ] softmax 를 쓰는 이유(합=1)를 안다
- [ ] 패딩 마스킹이 왜 필요한지 안다 ★
- [ ] 어텐션 층을 구현하고 `alpha` 합이 1인 것을 확인했다
- [ ] LSTM 과 LSTM+어텐션의 정확도를 비교했다
- [ ] **어텐션 가중치를 시각화하고 해석**했다 ★★
- [ ] Grad-CAM 과 어텐션 시각화의 공통 한계를 안다